# 문과생을 위한 주성분분석(PCA) — 학생 워크북 (실습)

빈칸 `____` 을 채워 셀을 완성한 뒤 실행한다. 막히면 각 셀 아래의 **▶ 정답 보기** 를 연다.
표는 `polars` 로 살펴본다.

## 0. 준비 — 이 셀은 그대로 실행한다 (글꼴 · 자료)

In [ ]:
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
for _f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
           "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"]:
    try:
        fm.fontManager.addfont(_f)
        plt.rcParams["font.family"] = fm.FontProperties(fname=_f).get_name(); break
    except Exception:
        pass
plt.rcParams["axes.unicode_minus"] = False

import numpy as np, polars as pl
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

iris = load_iris()
print("자료 크기:", iris.data.shape)   # (150, 4)

## 1. 자료를 polars 표로 보기 (그대로 실행)

In [ ]:
cols = ["꽃받침길이", "꽃받침너비", "꽃잎길이", "꽃잎너비"]
df = pl.DataFrame(iris.data, schema=cols)
df = df.with_columns(pl.Series("품종", [iris.target_names[i] for i in iris.target]))
df.head()

## 2. (빈칸 1) 표준화 — 네 변수를 같은 척도로

평균을 빼고 표준편차로 나누는 **표준화 도구**의 이름을 채운다.

In [ ]:
Z = ____().fit_transform(iris.data)
print(Z.mean(axis=0).round(2))   # [0. 0. 0. 0.]
print(Z.std(axis=0).round(2))    # [1. 1. 1. 1.]

<details><summary>▶ 정답 보기</summary>

```python
Z = StandardScaler().fit_transform(iris.data)
```

</details>

## 3. (빈칸 2·3) 방향 찾기 — PCA 를 자료에 맞추고 정보 비율 보기

빈칸 1 → 자료에 맞추는 메서드, 빈칸 2 → 각 PC 의 설명 비율을 담은 속성.

In [ ]:
pca = PCA().____(Z)
print(pca.________________.round(4))
# [0.7296 0.2285 0.0367 0.0052]

<details><summary>▶ 정답 보기</summary>

```python
pca = PCA().fit(Z)
print(pca.explained_variance_ratio_.round(4))
```

</details>

## 4. (빈칸 4) PC 스코어 — 각 꽃을 새 좌표로 옮기기

자료를 새 좌표(PC) 값으로 옮기는 메서드를 채운다. 결과를 polars 표로 본다.

In [ ]:
scores = pca.____(Z)
sc = pl.DataFrame(scores[:, :2], schema=["PC1", "PC2"])
sc = sc.with_columns(pl.Series("품종", [iris.target_names[i] for i in iris.target]))
sc.head()

<details><summary>▶ 정답 보기</summary>

```python
scores = pca.transform(Z)
```

</details>

## 5. (빈칸 5) 누적 정보 비율 — 두 방향이 몇 %를 채우나

위에서부터 정보 비율을 더해 가는 **누적합 함수**를 채운다.

In [ ]:
print(np.______(pca.explained_variance_ratio_).round(3))
# [0.73 0.958 0.995 1.   ]  → 앞쪽 두 방향이 약 96%

<details><summary>▶ 정답 보기</summary>

```python
print(np.cumsum(pca.explained_variance_ratio_).round(3))
```

</details>

## 6. (빈칸 6) 결과 그림 — PC1·PC2 평면에 세 품종 그리기

품종별로 점을 찍는다. 가로축 `scores[m, 0]`, 세로축은 `scores[m, ___]` 의 빈칸을 채운다(PC2 이므로 1).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cmap = ["#1F3A5F", "#C0392B", "#2A8C82"]; marks = ["o", "s", "^"]
for k, name in enumerate(iris.target_names):
    m = iris.target == k
    ax.scatter(scores[m, 0], scores[m, ____], c=cmap[k], marker=marks[k],
               s=34, alpha=0.8, edgecolor="white", label=name)
ax.set_xlabel("PC1 — 꽃 전체 크기 (73%)"); ax.set_ylabel("PC2 — 꽃받침 너비 (23%)")
ax.legend(); ax.set_title("두 새 좌표만으로 세 품종이 나뉜다"); plt.show()

<details><summary>▶ 정답 보기</summary>

```python
ax.scatter(scores[m, 0], scores[m, 1], c=cmap[k], marker=marks[k],
           s=34, alpha=0.8, edgecolor="white", label=name)
```

빈칸은 **1** 이다(PC2 는 둘째 열이므로 색인 1).

</details>

## 7. (빈칸 7) 새 축의 정체 — 각 변수의 기여도

각 변수가 PC1·PC2 에 기여하는 정도를 담은 **속성 이름**을 채운다.

In [ ]:
load = pl.DataFrame({
    "변수": cols,
    "PC1_기여": pca.__________[0].round(3),
    "PC2_기여": pca.__________[1].round(3),
})
load

<details><summary>▶ 정답 보기</summary>

```python
"PC1_기여": pca.components_[0].round(3),
"PC2_기여": pca.components_[1].round(3),
```

PC1 은 **꽃 전체 크기**, PC2 는 사실상 **꽃받침 너비** 축이다.

</details>